# Confidence Routing and Threshold Configuration

One of Portiere's core design principles is **confidence-based routing**: every mapping decision (both schema mappings and concept mappings) is assigned a confidence score, and that score determines the automated workflow path.

The routing tiers are:

| Confidence Range | Action | Description |
|-----------------|--------|-------------|
| >= auto_accept | **Auto-accept** | High confidence. Mapping is accepted automatically with no human review. |
| >= needs_review and < auto_accept | **Needs review** | Moderate confidence. Flagged for human verification before acceptance. |
| < needs_review | **Manual** | Low confidence. Requires manual intervention -- the system cannot reliably determine the correct mapping. |

This notebook demonstrates how to configure these thresholds for both schema mapping and concept mapping, and how the routing affects the review workflow.

## 1. Default Thresholds

Portiere ships with sensible defaults tuned for biomedical data mapping to OMOP CDM v5.4.

In [1]:
from portiere import PortiereConfig, ThresholdsConfig

# Create a config with default thresholds
default_config = PortiereConfig()

t = default_config.thresholds
print("Schema Mapping Thresholds:")
print(f"  Auto-accept: >= {t.schema_mapping.auto_accept}")
print(f"  Needs review: >= {t.schema_mapping.needs_review}")
print(f"  Manual: < {t.schema_mapping.needs_review}")
print()
print("Concept Mapping Thresholds:")
print(f"  Auto-accept: >= {t.concept_mapping.auto_accept}")
print(f"  Needs review: >= {t.concept_mapping.needs_review}")
print(f"  Manual: < {t.concept_mapping.needs_review}")
print()
print("Validation Thresholds:")
print(f"  Min completeness: {t.validation.min_completeness}")
print(f"  Min conformance: {t.validation.min_conformance}")
print(f"  Min plausibility: {t.validation.min_plausibility}")

Schema Mapping Thresholds:
  Auto-accept: >= 0.95
  Needs review: >= 0.7
  Manual: < 0.7

Concept Mapping Thresholds:
  Auto-accept: >= 0.95
  Needs review: >= 0.7
  Manual: < 0.7

Validation Thresholds:
  Min completeness: 0.95
  Min conformance: 0.98
  Min plausibility: 0.9


## Shared Athena Vocabulary Setup

Build (or reuse) BM25s and FAISS indexes from the Athena vocabulary download.
On the first run this takes ~10-30 minutes for FAISS embedding; subsequent runs
reuse the cached indexes.

In [2]:
from pathlib import Path

ATHENA_DIR   = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR    = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

from portiere.config import SchemaMappingThresholds, ConceptMappingThresholds, ValidationThresholds
from portiere import PortiereConfig, ThresholdsConfig, KnowledgeLayerConfig

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
    thresholds=ThresholdsConfig(
        schema_mapping=SchemaMappingThresholds(
            auto_accept=0.85,
            needs_review=0.60,
        ),
        concept_mapping=ConceptMappingThresholds(
            auto_accept=0.90,
            needs_review=0.65,
        ),
        validation=ValidationThresholds(
            min_completeness=0.90,
            min_conformance=0.95,
            min_plausibility=0.85,
        ),
    )
)

t = config.thresholds
print("Custom Schema Mapping Thresholds:")
print(f"  Auto-accept: >= {t.schema_mapping.auto_accept}")
print(f"  Needs review: >= {t.schema_mapping.needs_review}")
print()
print("Custom Concept Mapping Thresholds:")
print(f"  Auto-accept: >= {t.concept_mapping.auto_accept}")
print(f"  Needs review: >= {t.concept_mapping.needs_review}")
print()
print("Custom Validation Thresholds:")
print(f"  Min completeness: {t.validation.min_completeness}")
print(f"  Min conformance: {t.validation.min_conformance}")
print(f"  Min plausibility: {t.validation.min_plausibility}")

Custom Schema Mapping Thresholds:
  Auto-accept: >= 0.85
  Needs review: >= 0.6

Custom Concept Mapping Thresholds:
  Auto-accept: >= 0.9
  Needs review: >= 0.65

Custom Validation Thresholds:
  Min completeness: 0.9
  Min conformance: 0.95
  Min plausibility: 0.85


## 2. Custom Thresholds

You can tune thresholds to match your project's risk tolerance. Lowering the auto-accept threshold increases automation but may introduce more errors. Raising it reduces errors but increases the manual review burden.

Below is an example of a more permissive configuration suitable for an initial exploratory mapping pass.

In [3]:
from portiere.config import SchemaMappingThresholds, ConceptMappingThresholds, ValidationThresholds
from portiere import PortiereConfig, ThresholdsConfig

config = PortiereConfig(
    thresholds=ThresholdsConfig(
        schema_mapping=SchemaMappingThresholds(
            auto_accept=0.85,
            needs_review=0.60,
        ),
        concept_mapping=ConceptMappingThresholds(
            auto_accept=0.90,
            needs_review=0.65,
        ),
        validation=ValidationThresholds(
            min_completeness=0.90,
            min_conformance=0.95,
            min_plausibility=0.85,
        ),
    )
)

t = config.thresholds
print("Custom Schema Mapping Thresholds:")
print(f"  Auto-accept: >= {t.schema_mapping.auto_accept}")
print(f"  Needs review: >= {t.schema_mapping.needs_review}")
print()
print("Custom Concept Mapping Thresholds:")
print(f"  Auto-accept: >= {t.concept_mapping.auto_accept}")
print(f"  Needs review: >= {t.concept_mapping.needs_review}")
print()
print("Custom Validation Thresholds:")
print(f"  Min completeness: {t.validation.min_completeness}")
print(f"  Min conformance: {t.validation.min_conformance}")
print(f"  Min plausibility: {t.validation.min_plausibility}")

Custom Schema Mapping Thresholds:
  Auto-accept: >= 0.85
  Needs review: >= 0.6

Custom Concept Mapping Thresholds:
  Auto-accept: >= 0.9
  Needs review: >= 0.65

Custom Validation Thresholds:
  Min completeness: 0.9
  Min conformance: 0.95
  Min plausibility: 0.85


## 3. Schema Mapping with Confidence Routing

When you run `project.map_schema()`, each source column is matched against OMOP CDM target tables and columns. The confidence score determines which routing bucket the mapping falls into.

In [4]:
import portiere
from portiere.engines import PolarsEngine

# Initialize with custom thresholds
# The vocabularies parameter controls which standard vocabularies are searched
# during concept mapping. It defaults to ["SNOMED", "LOINC", "RxNorm", "ICD10CM"].
project = portiere.init(
    name="confidence_routing_demo",
    engine=PolarsEngine(),
    config=config,
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
)

# Ingest a sample source
source = project.add_source("example_assets/05_confidence_routing/sample_patients.csv")

# Run schema mapping
schema_map = project.map_schema(source)

2026-04-16 00:26:46 [info     ] PolarsEngine initialized      


2026-04-16 00:26:46 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


2026-04-16 00:26:46 [info     ] project.source_added           project=confidence_routing_demo source=sample_patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:26:49 [info     ] gx_profiler.profiled           columns=11 rows=15 source=sample_patients


...

2026-04-16 00:26:49 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:26:51 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:26:51 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:26:51 [info     ] local_storage.schema_mapping_saved items_count=11 project=confidence_routing_demo


2026-04-16 00:26:51 [info     ] project.schema_mapped          auto=10 project=confidence_routing_demo total=11


## 4. Schema Mapping Summary and Routing Breakdown

The `summary()` method provides a high-level view of how mappings were distributed across routing tiers.

In [5]:
summary = schema_map.summary()
print("Schema Mapping Summary:")
print(f"  Total columns: {summary['total']}")
print(f"  Auto-accepted: {summary['auto_accepted']}")
print(f"  Needs review: {summary['needs_review']}")
print(f"  Unmapped: {summary['unmapped']}")
print(f"  Auto-accept rate: {summary['auto_rate']:.1%}")
print()

# Inspect items by routing tier
print("--- Auto-Accepted Mappings ---")
for item in schema_map.auto_accepted():
    print(f"  {item.source_column} -> {item.target_table}.{item.target_column} "
          f"(confidence: {item.confidence:.2f})")

print()
print("--- Needs Review ---")
for item in schema_map.needs_review():
    print(f"  {item.source_column} -> {item.target_table}.{item.target_column} "
          f"(confidence: {item.confidence:.2f}, status: {item.status.value})")

Schema Mapping Summary:
  Total columns: 11
  Auto-accepted: 10
  Needs review: 0
  Unmapped: 1
  Auto-accept rate: 9090.9%

--- Auto-Accepted Mappings ---
  patient_id -> person.person_id (confidence: 0.95)
  first_name -> person.person_source_value (confidence: 0.95)
  date_of_birth -> person.birth_datetime (confidence: 0.95)
  gender -> person.gender_concept_id (confidence: 0.95)
  race -> person.race_concept_id (confidence: 0.95)
  ethnicity -> person.ethnicity_concept_id (confidence: 0.95)
  address -> location.address_1 (confidence: 0.95)
  city -> location.city (confidence: 0.95)
  state -> location.state (confidence: 0.95)
  zip_code -> location.zip (confidence: 0.95)

--- Needs Review ---


## 5. Concept Mapping with Confidence Routing

Concept mapping follows the same routing pattern. Each source code is matched against standard vocabulary concepts, and the confidence score determines the workflow path.

In [6]:
# Run concept mapping on a set of ICD-10 codes
concept_map = project.map_concepts(
    codes=["E11.9", "I10", "J44.1", "N18.3", "F32.9", "Z87.891", "R51"],
    vocabularies=["SNOMED", "ICD10CM"],
)

summary = concept_map.summary()
print("Concept Mapping Summary:")
print(f"  Total codes: {summary['total']}")
print(f"  Auto-mapped: {summary['auto_mapped']}")
print(f"  Needs review: {summary['needs_review']}")
print(f"  Manual required: {summary['manual_required']}")
print(f"  Auto-map rate: {summary['auto_rate']:.1%}")
print(f"  Coverage: {summary['coverage']:.1%}")

2026-04-16 00:26:51 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=None


2026-04-16 00:26:51 [warning  ] local_concept_mapper.no_knowledge_layer message='No knowledge_layer configured. Concept search will not work.'


2026-04-16 00:26:51 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=None llm_verifier=False reranker=True


2026-04-16 00:26:51 [warning  ] local_concept_mapper.no_backend query=E11.9


2026-04-16 00:26:51 [warning  ] local_concept_mapper.no_backend query=I10


...

2026-04-16 00:26:51 [warning  ] local_concept_mapper.no_backend query=R51


2026-04-16 00:26:51 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=7


2026-04-16 00:26:51 [info     ] local_storage.concept_mapping_saved items_count=7 project=confidence_routing_demo


2026-04-16 00:26:51 [info     ] project.concepts_mapped        auto_rate=0.0 project=confidence_routing_demo total=7


Concept Mapping Summary:
  Total codes: 7
  Auto-mapped: 0
  Needs review: 0
  Manual required: 7
  Auto-map rate: 0.0%
  Coverage: 0.0%


## 6. Review Workflow: Approve, Reject, Override

Items routed to the "needs review" tier must be resolved before the mapping can be finalized. Portiere provides three actions for each item:

- **approve()** -- Accept the top-ranked candidate (or a specific candidate by index). Sets method to AUTO.
- **reject()** -- Mark the item as unmapped. Sets method to UNMAPPED.
- **override()** -- Manually specify the correct target concept. Sets method to OVERRIDE.

In [7]:
# Review items that need human verification
review_items = concept_map.needs_review()
print(f"Items requiring review: {len(review_items)}")
print()

for item in review_items:
    print(f"Source code: {item.source_code}")
    print(f"  Description: {item.source_description}")
    print(f"  Top candidate: {item.target_concept_name} (ID: {item.target_concept_id})")
    print(f"  Confidence: {item.confidence:.2f}")
    print(f"  Method: {item.method.value}")
    if item.candidates:
        print(f"  All candidates:")
        for i, c in enumerate(item.candidates[:3]):
            print(f"    [{i}] {c.concept_name} (ID: {c.concept_id}, score: {c.score:.2f})")
    print()

Items requiring review: 0



In [8]:
# Demonstrate the three review actions
if len(review_items) >= 1:
    # Approve the first item with its top candidate
    review_items[0].approve(candidate_index=0)
    print(f"Approved: {review_items[0].source_code} -> {review_items[0].target_concept_name}")
    print(f"  Method is now: {review_items[0].method.value}")
    print()

if len(review_items) >= 2:
    # Reject the second item (mark as unmapped)
    review_items[1].reject()
    print(f"Rejected: {review_items[1].source_code}")
    print(f"  Method is now: {review_items[1].method.value}")
    print()

if len(review_items) >= 3:
    # Override the third item with a manually specified concept
    review_items[2].override(
        concept_id=443614,
        concept_name="Type 2 diabetes mellitus",
        vocabulary_id="SNOMED",
    )
    print(f"Overridden: {review_items[2].source_code} -> Type 2 diabetes mellitus")
    print(f"  Method is now: {review_items[2].method.value}")

## 7. Post-Review Summary Statistics

After resolving all review items, check the updated summary to confirm the mapping is ready for finalization.

In [9]:
# Approve all remaining review items that we have not yet handled
# In practice, you would review each one individually
concept_map.approve_all()

# Updated summary
summary = concept_map.summary()
print("Post-Review Concept Mapping Summary:")
print(f"  Total codes: {summary['total']}")
print(f"  Auto-mapped: {summary['auto_mapped']}")
print(f"  Needs review: {summary['needs_review']}")
print(f"  Manual required: {summary['manual_required']}")
print(f"  Auto-map rate: {summary['auto_rate']:.1%}")
print(f"  Coverage: {summary['coverage']:.1%}")
print()

# Check for any remaining unmapped items
unmapped = concept_map.unmapped()
if unmapped:
    print(f"Warning: {len(unmapped)} items still unmapped:")
    for item in unmapped:
        print(f"  - {item.source_code}: {item.source_description}")
else:
    print("All items have been mapped or reviewed.")

# Finalize the mapping
concept_map.finalize()
print("\nConcept mapping finalized.")

Post-Review Concept Mapping Summary:
  Total codes: 7
  Auto-mapped: 0
  Needs review: 0
  Manual required: 7
  Auto-map rate: 0.0%
  Coverage: 0.0%

  - E11.9: E11.9
  - I10: I10
  - J44.1: J44.1
  - N18.3: N18.3
  - F32.9: F32.9
  - Z87.891: Z87.891
  - R51: R51

Concept mapping finalized.


## Best Practices for Threshold Tuning

### Start with defaults, then adjust

The default thresholds (schema: 0.90/0.70, concept: 0.95/0.70) are calibrated for biomedical data mapping to OMOP CDM. Start with these and adjust based on your observed error rates.

### Consider your data quality

- **Clean, well-coded data** (e.g., structured EHR exports with standard ICD-10 codes): You can safely lower thresholds because the source data closely matches standard vocabularies.
- **Noisy, free-text data** (e.g., clinical notes, legacy systems with custom codes): Keep thresholds high to avoid propagating incorrect mappings.

### Balance automation vs. accuracy

| Scenario | Recommended auto_accept | Recommended needs_review |
|----------|------------------------|-------------------------|
| Exploratory analysis (errors tolerable) | 0.80 | 0.50 |
| Research study (moderate accuracy) | 0.90 | 0.70 |
| Clinical production (high accuracy required) | 0.95 | 0.80 |
| Regulatory submission (near-zero error tolerance) | 0.98 | 0.90 |

### Validation thresholds

Validation thresholds (`min_completeness`, `min_conformance`, `min_plausibility`) apply to the final ETL output and control whether the generated data passes quality checks:

- **Completeness** measures how many required fields are populated.
- **Conformance** measures how well values match expected data types, formats, and value sets.
- **Plausibility** measures whether values fall within clinically reasonable ranges.

For production ETL pipelines, keep these above 0.90. For exploratory work, you may relax them to focus on the mapping phase first.

### Iterative refinement

1. Run an initial mapping pass with default thresholds.
2. Review the summary statistics -- if auto_rate is too low, consider lowering auto_accept.
3. Spot-check the auto-accepted mappings for errors -- if error rate is unacceptable, raise auto_accept.
4. Adjust and re-run until you find the right balance for your dataset.